In [14]:
import os

# Set these before importing TensorFlow for cleaner logs and more reproducibility.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import random
import tarfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import random
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Activation

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("TensorFlow:", tf.__version__)
print("NumPy:     ", np.__version__)


TensorFlow: 2.20.0
NumPy:      2.0.2


In [15]:
# load file

path = "/content/drive/MyDrive/peptides.csv"
data = pd.read_csv(path)
data.head()

,FASTA,Label
0,FIHHIIGWISHGVRAIHRAIH,1
1,KWKLFKKGIGAVLKV,1
2,FRFKIKFRLKFRFKARFKFRAKFRA,1
3,GIKAKIIIKIKK,1
4,GLLSRLRDFLSDRGRRLGEKIERIGQKIKDLSEFFQS,1


In [16]:
data.columns
raw_x = np.array(data["FASTA"].astype(str))
raw_y = np.array(data["Label"].astype(int))

In [17]:
multi_cnn_roc_list = []
multi_cnn_pr_list = []

In [18]:
seq_len = 50 # all peptides padded/truncated to 50 tokens for fixed shape
vocab_size = 21 # keep only 21 most frequent peptides (should cover all)
embed_dim = 96
batch_size = 64
epochs = 20
patience = 4

In [19]:
# set seeds
def set_seeds(seed):
    """Seed Python, NumPy, and TensorFlow."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    if hasattr(tf.keras.utils, "set_random_seed"):
        tf.keras.utils.set_random_seed(seed)

In [20]:
def build_multi_cnn1d_(seq_len, vocab_size, embed_dim, filters=96, steps_p_epoch=100):
    inputs = keras.Input(shape=(1,),dtype=tf.string)

    # Adapt vectorizer on training partition only
    vectorize = tf.keras.layers.TextVectorization(
      max_tokens=vocab_size,
      output_mode="int", #gives integer IDs per token
      output_sequence_length=seq_len,
      standardize=None,
      split="character")

    x = vectorize(inputs)

    x = tf.keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(x)

    # Branches
    b1 = layers.Conv1D(filters, 3, padding="same", activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    b2 = layers.Conv1D(filters, 5, padding="same", activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    b3 = layers.Conv1D(filters, 7, padding="same", activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-5))(x)

    x = layers.Concatenate(name= "multi_kernel_features")([b1, b2, b3])
    x = layers.LayerNormalization()(x)
    x = layers.Conv1D(192, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.LayerNormalization()(x)

    # Pooling
    gap = layers.GlobalAveragePooling1D()(x)
    gmp = layers.GlobalMaxPooling1D()(x)
    last = layers.Lambda(lambda z: z[:, -1, :], name="last_cnn_state")(x)

    x = layers.Concatenate()([gap, gmp, last])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    # Learning Rate Schedule
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-3,
        decay_steps=epochs*steps_p_epoch,
        alpha=0.1)

    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule)

    model = keras.Model(inputs, outputs, name="peptides_1dcnn")
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model, vectorize

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
for seed in [1,2,3,4,5]:
  set_seeds(seed)
  # 90% train / 10% test split
  x_train, x_test, y_train, y_test = train_test_split(
      raw_x, raw_y, test_size=0.1, random_state=seed, stratify=raw_y)

  val_size = int(0.1 * len(x_train))
  x_tr, x_val = x_train[:-val_size], x_train[-val_size:]
  y_tr, y_val = y_train[:-val_size], y_train[-val_size:]

  model, vectorize = build_multi_cnn1d_(seq_len, vocab_size, embed_dim, filters=96)
  vectorize.adapt(x_tr)

  # Build tf.data.Datasets for robust string input to model.fit()
  AUTO = tf.data.AUTOTUNE
  train_ds = ((tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
            .shuffle(len(x_tr), seed=seed)).batch(batch_size).prefetch(AUTO))
  val_ds = ((tf.data.Dataset.from_tensor_slices((x_val, y_val))
            .batch(batch_size).prefetch(AUTO)))

  callbacks = [tf.keras.callbacks.EarlyStopping(
      monitor="val_loss", patience=patience, min_delta=1e-4,
      restore_best_weights=True)]

  history = model.fit(
      train_ds, validation_data=val_ds, epochs=15,
      callbacks=callbacks, verbose=0)

  test_ds = (tf.data.Dataset.from_tensor_slices((x_test, y_test))
            .batch(batch_size).prefetch(AUTO))

  y_prob = model.predict(test_ds).ravel()
  multi_cnn_roc_list.append(roc_auc_score(y_test, y_prob))
  multi_cnn_pr_list.append(average_precision_score(y_test, y_prob))

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [11]:
results = pd.DataFrame({
    "seed": [1,2,3,4,5],
    "roc_auc": multi_cnn_roc_list,
    "pr_auc": multi_cnn_pr_list
})

results

,seed,roc_auc,pr_auc
0,1,0.987955,0.988637
1,2,0.984273,0.984840
2,3,0.986034,0.987042
3,4,0.986096,0.986702
4,5,0.986515,0.987361


In [12]:
print("Multi-Kernel 1D CNN ROC AUC:", np.mean(multi_cnn_roc_list), np.std(multi_cnn_roc_list))
print("Multi-Kernel 1D CNN PR  AUC:", np.mean(multi_cnn_pr_list),  np.std(multi_cnn_pr_list))



Multi-Kernel 1D CNN ROC AUC: 0.9861748001059251 0.001177001872335388
Multi-Kernel 1D CNN PR  AUC: 0.9869162293575044 0.0012272679832234113


In [13]:
print(f"The Multi‑Kernel 1D CNN achieved...")
print(f"... a mean ROC AUC of {f'{np.mean(multi_cnn_roc_list):.4f}'} with a standard deviation of {f'{np.std(multi_cnn_roc_list):.4f}'}.")
print(f"... a mean PR  AUC of {f'{np.mean(multi_cnn_pr_list):.4f}'} with a standard deviation of {f'{np.std(multi_cnn_pr_list):.4f}'}.")


The Multi‑Kernel 1D CNN achieved...
... a mean ROC AUC of 0.9862 with a standard deviation of 0.0012.
... a mean PR  AUC of 0.9869 with a standard deviation of 0.0012.


#Discussion

From this project, I learned that complexity is not an instant fix, and models need to be carefully considered. For example, I began with a TF-IDF + Logistic Regression baseline which performed best compared to a simple Text Vectorization + Embedding + GAP model, but neither captured the local sequence motifs adequately. I thought to jump to a multi-kernel CNN model which, to my surprise, performed worse. I found better luck building a decent 1D CNN model instead, optimising it and tuning hyperparameters with hyperband, before progressing to a multi-kernel CNN model. Optimizaions were, for example, inclusion of dropout, l2 regularisation + AdamW, and a cosine-decay learning rate schedule. These stabilised traning across seeds and improved generalisation.  With these changes, this multi-kernel model triumphed over other models and met all target criteria.